# Global multi-asset portfolio research

**Question:** does optimized allocation improve the balance of growth, risk and trading
effort relative to a simple global 60/40 policy?

This notebook is a guided research note. It reads one published evidence bundle, so its
numbers match the dashboard and executive report. We first understand the reference,
then inspect what the models actually delivered. All returns are in USD.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root / "src"))

from portfolio_management_project.presentation.evidence import Evidence
from portfolio_management_project.presentation.language import Language
from portfolio_management_project.presentation.notebook import show_table
from portfolio_management_project.presentation.charts import allocations, lines

evidence = Evidence(root)
text = Language(root, 'en')
performance = evidence.table("strategy_performance")
wealth = evidence.dates("normalized_wealth")
print(f"{wealth.index.min():%Y-%m-%d} → {wealth.index.max():%Y-%m-%d}")

## 1. What can the portfolio hold?

The nine ETFs cover global equities, nominal and inflation-linked bonds, investment-grade
credit, real estate, gold and cash. The economic role matters more than the ticker count.
A diversified list can still share exposure to equity risk or interest rates.

In [ ]:
import pandas as pd
universe = pd.DataFrame([
    {"Ticker": asset.ticker, "Role": text(asset.role)}
    for asset in evidence.config.universe.assets
])
show_table(universe, text)

## 2. Understand the benchmark before judging a model

The reference holds 60% VT (global equities) and 40% BND (US investment-grade bonds).
It resets those weights quarterly; holdings drift between rebalances. This is an external
policy benchmark, not an optimizer with the same ETF universe or 35% cap.

Both benchmark and strategies pay 10 bps on gross rebalance notional. Initial purchase costs
are excluded for both. BIL is a separate cash reference for Sharpe, not the 60/40 benchmark.

In [ ]:
display(show_table(performance.loc[["benchmark_60_40"]], text, strategies=True))
lines(wealth[["benchmark_60_40"]], text, "Growth of 100 USD")

## 3. Check the data before interpreting the model

Adjusted closing prices include adjustments for distributions and splits. Inspect missing
observations and historical correlations; a clean matrix does not remove provider revisions,
survivorship bias or changing relationships during stress.

In [ ]:
display(show_table(evidence.table("data_quality"), text))
show_table(evidence.table("correlations"), text)

## 4. What does each allocation rule try to do?

Equal weight is the simple control. Minimum variance emphasizes stability. Mean–variance
trades expected return against risk. Target return minimizes risk for a return objective.
Maximum Sharpe seeks estimated excess return per unit of risk.

The chart below uses the full estimation sample. It explains the models; it is not a
walk-forward performance result. Inputs are historical mean returns and Ledoit–Wolf covariance.

In [ ]:
allocations(evidence.table("construction_weights").T, text)

## 5. Make the target-return adjustment visible

A 7% objective may be infeasible within a particular training window and the position limits.
The policy then uses the nearest feasible objective. This is an explicit rule, not a promised
realized return. Each row records the information date and requested/effective targets.

In [ ]:
audit = evidence.table("allocation_audit")
show_table(audit[audit.target_adjusted], text)

## 6. Follow the out-of-sample wealth paths

Every decision uses a trailing window ending before the decision day. Implementation occurs
at that day's close, and the allocation earns subsequent returns. The baseline uses three
years of daily observations. Calendar versions trade monthly or quarterly. Band versions
refresh targets monthly and trade only when a weight deviates by more than ±2.5, ±5 or
±7.5 percentage points. Crossing a band restores the complete target and incurs costs.

Compare the full path, not only the endpoint. A smoother path can still finish below the
benchmark; that is a trade-off to explain, not something to hide.

In [ ]:
selected = ["equal_weight_quarterly", "target_return_quarterly", "target_return_band_500bp", "benchmark_60_40"]
lines(wealth[selected], text, "Growth of 100 USD")

## 7. Separate compound growth from risk-adjusted performance

CAGR describes compound growth. Realized Sharpe uses the annualized mean and sample
standard deviation of daily returns in excess of the dated BIL series. Both strategies and
benchmark use the same cash calendar. Maximum drawdown is a positive loss magnitude in tables.

In [ ]:
show_table(performance, text, strategies=True)

## 8. How painful were the losses?

Drawdown measures the distance below a previous peak. Read depth together with time to
recovery. An open episode has no recovery date; durations are calendar days. A low annual
volatility does not mean an investor avoids long periods below a previous high.

In [ ]:
lines(evidence.dates("strategy_drawdowns")[selected], text, "Underwater performance", percent=True)

## 9. Inspect what the strategy held and traded

Decision weights reveal how the rule responds to changing estimates. The step chart shows
targets, not daily drift. Turnover is accumulated over the full evaluation period; transaction
cost is measured in the same wealth units as the initial 100.

In [ ]:
weights = evidence.table("decision_weights", 2).loc["target_return_quarterly"]
weights.index = pd.to_datetime(weights.index)
display(show_table(evidence.table("strategy_trading"), text, strategies=True))
allocations(weights, text, history=True)

## 10. Challenge the result

Window scenarios start later, in 2013, so even five years of training fit before evaluation.
All windows share this calendar. Compare them with each other, not directly with the longer
baseline. Uncompleted optimizations remain in the evidence with a status and a reason.

In [ ]:
sensitivity = evidence.table("lookback_sensitivity", 2)
show_table(sensitivity[["status", "start_date", "end_date", "annualized_return", "sharpe_ratio", "maximum_drawdown"]], text)

## 11. Use the scorecard as a preference, not a proof

The scorecard gives equal rank weight to CAGR, volatility, Sharpe, drawdown, Calmar, turnover,
information ratio and average largest position. Numerical ties are rounded to ten decimals.
Several measures overlap. The lowest average rank is the leader under these preferences;
it is not a universal optimum or an independently validated prediction.

In [ ]:
scores = evidence.table("strategy_scorecard")
show_table(scores[["overall_rank", "balanced_rank_score", "annualized_return", "sharpe_ratio", "maximum_drawdown"]], text, strategies=True)

## 12. Close with the trade-off

Compare the scorecard leader with the benchmark below. Discuss growth, volatility and loss
experience together. Choosing a winner after seeing this sample is still exploratory. The
next step is an independent holdout or a prospective paper portfolio.

ETF selection reflects present-day availability. The experiment excludes taxes, market
impact, changing spreads, investor liabilities and currency hedging. These limits are part
of the conclusion, not footnotes that make the comparison disappear.

In [ ]:
leader = scores.index[0]
show_table(performance.loc[[leader, "benchmark_60_40"]], text, strategies=True)

## Rebuild the research

The displayed tables come from the same publication bundle as the dashboard. To change
assumptions, edit the YAML inputs and run the command below from the project root. This
calls the core module through the project pipelines; it does not download fresh data.

```bash
python scripts/run_study.py
```

To request a new provider download first, run `python scripts/build_data.py`. Record the
new data manifest before interpreting differences from an earlier run.